In [1]:
# Project path setup: works whether Jupyter was launched from the project root or notebooks/.
from pathlib import Path
import os
cwd = Path.cwd()
if (cwd / "data").is_dir() and (cwd / "notebooks").is_dir():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data").is_dir() and cwd.name == "notebooks":
    PROJECT_ROOT = cwd.parent
else:
    raise RuntimeError("Open this notebook from the project root or its notebooks/ folder.")
os.chdir(PROJECT_ROOT / "notebooks")
print("Project root:", PROJECT_ROOT)
print("Notebook working directory:", Path.cwd())

Project root: /home/akshat/Desktop/main files (i think)/ml course work/ml project
Notebook working directory: /home/akshat/Desktop/main files (i think)/ml course work/ml project/notebooks


# 10 — Final Comparison: Logistic Regression vs Compact LinearSVC

This notebook compares the selected Logistic Regression model (100k feature limit; actual vocabulary about 27,293) with the compact LinearSVC (10k features). Both use the same training split and the same stratified five-fold CV splits.

It reports training/CV performance, the existing held-out test metrics, serialized browser payload sizes, Python-side prediction latency and prediction agreement. The test split was inspected earlier in development, so test metrics are descriptive project results—not a pristine independent validation.

In [2]:
import gzip
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    balanced_accuracy_score, roc_auc_score, classification_report, confusion_matrix)

TRAIN_PATH = Path("../data/processed/train.csv")
TEST_PATH = Path("../data/processed/test.csv")
OUTPUT_DIR = Path("../results/final_model_comparison")
PAYLOAD_DIR = Path("../results/final_model_comparison/browser_payloads")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PAYLOAD_DIR.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
X_train = train_df["text"].fillna("").astype(str)
y_train = train_df["target"].astype(int)
X_test = test_df["text"].fillna("").astype(str)
y_test = test_df["target"].astype(int)
RANDOM_STATE = 42

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 32328
Testing samples: 8082


In [3]:

def create_vectorizer(max_features):
    return TfidfVectorizer(
        stop_words=None,
        ngram_range=(1, 2),
        min_df=10,
        max_df=1.0,
        max_features=max_features,
        sublinear_tf=True,
        lowercase=True,
        norm="l2",
        use_idf=True,
        smooth_idf=True,
        token_pattern=r"(?u)\b\w\w+\b"
    )


models = {
    "Logistic Regression (100k)": Pipeline([
        ("tfidf", create_vectorizer(100000)),
        ("classifier", LogisticRegression(
            C=3,
            penalty="l2",
            solver="liblinear",
            class_weight=None,
            max_iter=5000,
            random_state=RANDOM_STATE
        ))
    ]),

    "LinearSVC (10k)": Pipeline([
        ("tfidf", create_vectorizer(10000)),
        ("classifier", LinearSVC(
            C=1,
            class_weight=None,
            max_iter=10000,
            random_state=RANDOM_STATE
        ))
    ])
}


In [4]:

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

cv_results = {}

for name, model in models.items():
    print(f"\nRunning cross-validation: {name}")

    start = time.perf_counter()

    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring={
            "macro_f1": "f1_macro",
            "accuracy": "accuracy",
            "balanced_accuracy": "balanced_accuracy"
        },
        return_train_score=True,
        n_jobs=1
    )

    elapsed = time.perf_counter() - start

    cv_results[name] = {
        "CV Macro F1": scores["test_macro_f1"].mean(),
        "CV Macro F1 Std": scores["test_macro_f1"].std(),
        "Training Macro F1": scores["train_macro_f1"].mean(),
        "Generalisation Gap": (
            scores["train_macro_f1"].mean()
            - scores["test_macro_f1"].mean()
        ),
        "CV Accuracy": scores["test_accuracy"].mean(),
        "CV Balanced Accuracy": scores["test_balanced_accuracy"].mean(),
        "CV Time (seconds)": elapsed
    }

    print("CV Macro F1:", round(scores["test_macro_f1"].mean(), 5))
    print("CV Std:", round(scores["test_macro_f1"].std(), 5))
    print("Training Macro F1:", round(scores["train_macro_f1"].mean(), 5))
    print("Time:", round(elapsed, 2), "seconds")

cv_table = pd.DataFrame(cv_results).T
display(cv_table.round(5))

cv_table.to_csv(OUTPUT_DIR / "cross_validation_results.csv")



Running cross-validation: Logistic Regression (100k)
CV Macro F1: 0.93977
CV Std: 0.00232
Training Macro F1: 0.97941
Time: 18.15 seconds

Running cross-validation: LinearSVC (10k)
CV Macro F1: 0.93922
CV Std: 0.00083
Training Macro F1: 0.98975
Time: 16.86 seconds


,CV Macro F1,CV Macro F1 Std,Training Macro F1,Generalisation Gap,CV Accuracy,CV Balanced Accuracy,CV Time (seconds)
Logistic Regression (100k),0.93977,0.00232,0.97941,0.03964,0.93977,0.93977,18.14608
LinearSVC (10k),0.93922,0.00083,0.98975,0.05053,0.93922,0.93922,16.86441


In [5]:

fitted_models = {}
test_results = {}

for name, model in models.items():
    print(f"\nTraining final model: {name}")

    start = time.perf_counter()
    model.fit(X_train, y_train)
    fit_time = time.perf_counter() - start

    fitted_models[name] = model

    y_pred = model.predict(X_test)

    result = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision (Macro)": precision_score(
            y_test, y_pred, average="macro", zero_division=0
        ),
        "Recall (Macro)": recall_score(
            y_test, y_pred, average="macro", zero_division=0
        ),
        "Macro F1": f1_score(
            y_test, y_pred, average="macro", zero_division=0
        ),
        "Balanced Accuracy": balanced_accuracy_score(y_test, y_pred),
        "Fit Time (seconds)": fit_time
    }

    # Both models expose a continuous decision score.
    # For Logistic Regression, this is its log-odds decision score.
    # For LinearSVC, it is a margin, not a probability.
    decision_scores = model.decision_function(X_test)
    result["ROC-AUC"] = roc_auc_score(y_test, decision_scores)

    test_results[name] = result

    print(classification_report(
        y_test,
        y_pred,
        target_names=["OR (Authentic)", "CG (Computer Generated)"],
        zero_division=0
    ))

    print("Confusion matrix:")
    print(confusion_matrix(y_test, y_pred))

test_table = pd.DataFrame(test_results).T
display(test_table.round(5))

test_table.to_csv(OUTPUT_DIR / "test_results.csv")



Training final model: Logistic Regression (100k)
                         precision    recall  f1-score   support

         OR (Authentic)       0.94      0.95      0.95      4043
CG (Computer Generated)       0.95      0.93      0.94      4039

               accuracy                           0.94      8082
              macro avg       0.94      0.94      0.94      8082
           weighted avg       0.94      0.94      0.94      8082

Confusion matrix:
[[3860  183]
 [ 263 3776]]

Training final model: LinearSVC (10k)
                         precision    recall  f1-score   support

         OR (Authentic)       0.94      0.94      0.94      4043
CG (Computer Generated)       0.94      0.94      0.94      4039

               accuracy                           0.94      8082
              macro avg       0.94      0.94      0.94      8082
           weighted avg       0.94      0.94      0.94      8082

Confusion matrix:
[[3811  232]
 [ 249 3790]]


,Accuracy,Precision (Macro),Recall (Macro),Macro F1,Balanced Accuracy,Fit Time (seconds),ROC-AUC
Logistic Regression (100k),0.94482,0.94499,0.94481,0.94481,0.94481,2.84958,0.98706
LinearSVC (10k),0.94049,0.94049,0.94048,0.94048,0.94048,2.51801,0.98655


In [6]:

def export_browser_payload(model, output_path):
    vectorizer = model.named_steps["tfidf"]
    classifier = model.named_steps["classifier"]

    payload = {
        "vocabulary": {
            str(word): int(index)
            for word, index in vectorizer.vocabulary_.items()
        },
        "idf": [float(x) for x in vectorizer.idf_],
        "coefficients": [
            float(x) for x in classifier.coef_[0]
        ],
        "intercept": float(classifier.intercept_[0]),
        "classes": [int(x) for x in classifier.classes_],
        "ngram_range": list(vectorizer.ngram_range),
        "max_features": vectorizer.max_features,
        "model_type": classifier.__class__.__name__
    }

    json_string = json.dumps(
        payload,
        separators=(",", ":"),
        ensure_ascii=False
    )

    json_bytes = json_string.encode("utf-8")
    gzip_bytes = gzip.compress(json_bytes)

    with open(output_path, "wb") as f:
        f.write(json_bytes)

    return {
        "Vocabulary Size": len(vectorizer.vocabulary_),
        "JSON Size (MB)": len(json_bytes) / (1024 ** 2),
        "Gzip Size (MB)": len(gzip_bytes) / (1024 ** 2),
        "JSON Size (KB)": len(json_bytes) / 1024,
        "Gzip Size (KB)": len(gzip_bytes) / 1024
    }


size_results = {}

for name, model in fitted_models.items():
    filename = (
        "logistic_regression.json"
        if name.startswith("Logistic")
        else "linearsvc_10k.json"
    )

    output_path = PAYLOAD_DIR / filename

    size_results[name] = export_browser_payload(
        model,
        output_path
    )

size_table = pd.DataFrame(size_results).T
display(size_table.round(4))

size_table.to_csv(OUTPUT_DIR / "browser_model_sizes.csv")


,Vocabulary Size,JSON Size (MB),Gzip Size (MB),JSON Size (KB),Gzip Size (KB)
Logistic Regression (100k),27293.0,1.4380,0.4919,1472.5469,503.7109
LinearSVC (10k),10000.0,0.5128,0.1839,525.0732,188.2793


In [7]:

latency_results = {}

# Use the first 1000 test reviews for benchmarking
X_benchmark = X_test.iloc[:min(1000, len(X_test))]

for name, model in fitted_models.items():

    # Warm-up
    model.predict(X_benchmark.iloc[:10])

    start = time.perf_counter()
    predictions = model.predict(X_benchmark)
    elapsed = time.perf_counter() - start

    latency_results[name] = {
        "Reviews Tested": len(X_benchmark),
        "Total Prediction Time (seconds)": elapsed,
        "Average Time per Review (ms)": (
            elapsed / len(X_benchmark) * 1000
        ),
        "Reviews per Second": (
            len(X_benchmark) / elapsed
        )
    }

latency_table = pd.DataFrame(latency_results).T
display(latency_table.round(5))

latency_table.to_csv(OUTPUT_DIR / "prediction_latency.csv")


,Reviews Tested,Total Prediction Time (seconds),Average Time per Review (ms),Reviews per Second
Logistic Regression (100k),1000.0,0.04805,0.04805,20810.57826
LinearSVC (10k),1000.0,0.04431,0.04431,22568.89652


In [8]:

lr_predictions = fitted_models[
    "Logistic Regression (100k)"
].predict(X_test)

svc_predictions = fitted_models[
    "LinearSVC (10k)"
].predict(X_test)

agreement = np.mean(lr_predictions == svc_predictions)

print("Prediction agreement:", round(agreement * 100, 3), "%")
print("Disagreements:", np.sum(lr_predictions != svc_predictions))

disagreement_df = pd.DataFrame({
    "Review": X_test.reset_index(drop=True),
    "Actual": y_test.reset_index(drop=True),
    "Logistic Regression": lr_predictions,
    "LinearSVC": svc_predictions
})

disagreement_df = disagreement_df[
    disagreement_df["Logistic Regression"]
    != disagreement_df["LinearSVC"]
]

disagreement_df.to_csv(
    OUTPUT_DIR / "disagreement_examples.csv",
    index=False
)

display(disagreement_df.head(10))


Prediction agreement: 97.29 %
Disagreements: 219


,Review,Actual,Logistic Regression,LinearSVC
14,Kittie loves hers. Have to go back and forth t...,1,0,1
23,I do a lot of Homebrew. This pot is perfect. E...,0,1,0
40,This was a very nice book to read. I read it t...,0,1,0
70,The setting of the Amish and the people they l...,1,1,0
83,Gorgeous artwork. Great for a book club. Not...,1,0,1
195,LOVE this carrying case. Even better than the ...,1,1,0
225,Looks good. Affordable price. Easy to assemble...,1,1,0
229,"After trying 3 different models, this one it h...",0,0,1
309,Large capacity. Will keep stuff hot/cold for ...,0,1,0
329,"Complements different color schemes, that are ...",1,0,1


In [9]:

final_comparison = cv_table.join(
    test_table,
    lsuffix="_CV",
    rsuffix="_Test"
).join(size_table).join(latency_table)

display(final_comparison.round(5))

final_comparison.to_csv(
    OUTPUT_DIR / "final_comparison.csv"
)

print("\nAll comparison results saved in:", OUTPUT_DIR)
print("Browser JSON files saved in:", PAYLOAD_DIR)


,CV Macro F1,CV Macro F1 Std,Training Macro F1,Generalisation Gap,CV Accuracy,CV Balanced Accuracy,CV Time (seconds),Accuracy,Precision (Macro),Recall (Macro),...,ROC-AUC,Vocabulary Size,JSON Size (MB),Gzip Size (MB),JSON Size (KB),Gzip Size (KB),Reviews Tested,Total Prediction Time (seconds),Average Time per Review (ms),Reviews per Second
Logistic Regression (100k),0.93977,0.00232,0.97941,0.03964,0.93977,0.93977,18.14608,0.94482,0.94499,0.94481,...,0.98706,27293.0,1.43803,0.49191,1472.54688,503.71094,1000.0,0.04805,0.04805,20810.57826
LinearSVC (10k),0.93922,0.00083,0.98975,0.05053,0.93922,0.93922,16.86441,0.94049,0.94049,0.94048,...,0.98655,10000.0,0.51277,0.18387,525.07324,188.27930,1000.0,0.04431,0.04431,22568.89652



All comparison results saved in: ../results/final_model_comparison
Browser JSON files saved in: ../results/final_model_comparison/browser_payloads
